# Using a Custom Model



---
## 1. Setup & Configuration
Import necessary modules 

In [ ]:
# to run successfully the packages for jupyter notebook need to be installed:
# uv pip install ipykernel, ipython

from IPython.display import display
import os 
from pathlib import Path

# load the specific package
import bacpipe

Set the working directory to the repository root and clean the previous tests if needed/wanted

In [ ]:
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# replace this with the path to the directory on your machine
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# os.chdir('../..')
os.chdir('/media/haupert/data/mes_projets/_packages/bacpipe.git/bacpipe')

# Change the value of the key main_results_dir in the namespace bacpipe.settings to change the directory 
# where the results of the tutorials are stored. By default, it is set to './bacpipe_results'.
bacpipe.settings.main_results_dir = str(Path(bacpipe.settings.main_results_dir) / 'extending_an_existing_model')

# !WARNING! the following code deletes the folder where the results of this tutorial is stored to be sure to start with a clean folder. 
# If you have important data in this folder, please comment it before running this code.
folder_path = bacpipe.settings.main_results_dir
if os.path.exists(folder_path):
    # Prompt the user
    user_input = input(f"Are you sure you want to delete '{folder_path}'? (y/n): ").lower().strip()

    if user_input == 'y':
        import shutil
        shutil.rmtree(folder_path) 
        print(f"Folder {folder_path} deleted.")
    else:
        print("Operation cancelled.")

else:
    print(f"Folder {folder_path} not found.")

Set global constants that will use through the notebook.

In [ ]:
MODEL_NAME = 'custom_birdnet'           # Choose the name of the custom model to run.
AUDIO_DIR = 'bacpipe/tests/test_data'   # path to directory containing audio files

Set the working directory to the repository root and clean the previous tests if needed/wanted

---
## 2. Extending an Existing Model
Subclass an existing bacpipe model to modify its behaviour — for example, squaring the input before passing it through BirdNET.

In [ ]:
# !BUG I can't understand what is input (an array of melspec ready to be processed by the model) 

# Select the class Model of the desired model, here birdnet, but it could be another model in the list.
# To know all supported models, run bacpipe.supported_models.
# For instance, if one wants to use birdMAE, the code is :
# from bacpipe.model_pipelines.feature_extractors.birdmae import Model

from bacpipe.model_pipelines.feature_extractors.birdnet import Model

# Create a subclass from the class Model depending on the chosen model
class MyBirdNETModel(Model):
    def __call__(self, input):
        input = input
        return self.embeds(input, training=False)

---
## 3. Using this new built-in model

1. First generate the feature vectors (which are mels in this case, and not the embeddings of a deeplearning model)
2. Train a probe to evaluate the quality of the feature vectors (or embeddings) of the model

In [ ]:
# load the data to be process as well the model and compute the embeddings
loader_obj = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,                               # name of the model to run. Supported models are in bacpipe.supported_models
    audio_dir=AUDIO_DIR,                # path to directory containing audio files  
    CustomModel=MyBirdNETModel
)

# get the computed embeddings as an array
embeds = loader_obj.embeddings(return_type='array')

In [ ]:
# Run this function after computing the embeddings otherwise it is no able to find the connection between embeddings and labels
gt = bacpipe.ground_truth_by_model(
    model=MODEL_NAME, 
    audio_dir=AUDIO_DIR, 
    annotations_filename='annotations.csv',
    single_label=False,
    overwrite=False
)

# Train and test a new probe associated to the feature vectors of the model 
# and evaluate the performance of the probe. 
# The returned metrics are the same as for the evaluation of a classification model, 
# but here they are used to evaluate the quality of the embeddings of the model.
probe, label2idx, metrics = bacpipe.probing_pipeline(
    model_name=MODEL_NAME, 
    ground_truth=gt,
    embeds=embeds)

# display the main metrics of the probe
display(metrics['overall'])